In [42]:
import os
import sys
from dotenv import load_dotenv
from sklearn.metrics import adjusted_mutual_info_score
load_dotenv() 

# Set the target folder name you want to reach
target_folder = "NCEAS_Unsupervised_NLP"

# Get the current working directory
current_dir = os.getcwd()

# Loop to move up the directory tree until we reach the target folder
while os.path.basename(current_dir) != target_folder:
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    if parent_dir == current_dir:
        # If we reach the root directory and haven't found the target, exit
        raise FileNotFoundError(f"{target_folder} not found in the directory tree.")
    current_dir = parent_dir

# Change working directory to the project root
os.chdir(current_dir)

# Add the "NCEAS_Unsupervised_NLPt" directory to sys.path
sys.path.insert(0, current_dir)

In [43]:
import pandas as pd

In [44]:
data_sources = {
    "Qwen3-Embedding-0.6B": "Qwen3-Embedding-0.6B_results/qwen3_arxiv_results.csv",
    "all-MiniLM-L6-v2": "all-MiniLM-L6-v2_results/all_MiniLM_other_arxiv_results.csv"
}

In [45]:
import pandas as pd

# Toggle this
USE_COMBINED = True

if USE_COMBINED:
    # Load combined results (arXiv comparison file)
    df = pd.read_csv("src/evaluations/arxiv/results/arxiv_comparison.csv")

    # Standardize column names (just in case)
    df.columns = df.columns.str.strip()

    # Make sure column names match expected ones
    # (adjust if your CSV uses slightly different names)
    df = df.rename(columns={
        "embedding_model": "embedding",
        "reduction_method": "reduction",
        "cluster_method": "cluster_method"
    })

    # Group and average metrics
    final_df = (
        df.groupby(['embedding', 'reduction', 'cluster_method'])[['ARI', 'AMI']]
        .mean()
        .reset_index()
    )

else:
    results = []

    data_sources = {
        "MiniLM": "path_to_minilm.csv",
        "Qwen": "path_to_qwen.csv"
    }

    for model, filepath in data_sources.items():
        df = pd.read_csv(filepath)

        # Clean column names
        df.columns = df.columns.str.strip()

        # First aggregation (within level)
        grouped = (
            df.groupby(['reduction_method', 'cluster_method', 'level'])[['ARI', 'AMI']]
            .median()
            .reset_index()
        )

        # Then average across levels
        grouped = (
            grouped.groupby(['reduction_method', 'cluster_method'])[['ARI', 'AMI']]
            .mean()
            .reset_index()
        )

        grouped['embedding'] = model
        results.append(grouped)


In [46]:
final_df = pd.concat(results, ignore_index=True)

In [47]:
# Concatenate all results
final_df = pd.concat(results, ignore_index=True)

# Optional: display or save
final_df=final_df.replace({"DC":"Diffusion condensation"})
final_df=final_df.replace({"Diffusion Condensation":"Diffusion condensation"})
final_df = final_df[final_df['reduction_method']!="BASE-PHATE"]
final_df = final_df[final_df['reduction_method']!="None"]
final_df = final_df[final_df['reduction_method']!="tSNE"]
final_df

,reduction_method,cluster_method,ARI,embedding_model
0,PCA,Agglomerative,0.478989,Qwen3-Embedding-0.6B
1,PCA,Diffusion condensation,0.082603,Qwen3-Embedding-0.6B
2,PCA,HDBSCAN,0.000018,Qwen3-Embedding-0.6B
3,PHATE,Agglomerative,0.476800,Qwen3-Embedding-0.6B
4,PHATE,Diffusion condensation,0.296899,Qwen3-Embedding-0.6B
5,PHATE,HDBSCAN,-0.000139,Qwen3-Embedding-0.6B
6,UMAP,Agglomerative,0.490207,Qwen3-Embedding-0.6B
7,UMAP,Diffusion condensation,0.112346,Qwen3-Embedding-0.6B
8,UMAP,HDBSCAN,0.428488,Qwen3-Embedding-0.6B
9,PCA,Agglomerative,0.263159,all-MiniLM-L6-v2


In [48]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
final_df[final_df['cluster_method']=="Diffusion Condensation"].sort_values(by='ARI',ascending= False)


,reduction_method,cluster_method,ARI,embedding_model


In [49]:
final_df = final_df.rename(columns={
    "embedding_model": "embedding",
    "reduction_method": "reduction"
})

In [50]:
import matplotlib.pyplot as plt

df = final_df.copy()

reduction_order = ['PHATE', 'PCA', 'UMAP']
cluster_order = ['Agglomerative', 'HDBSCAN', 'Diffusion condensation']
embedding_order = ['Qwen3-Embedding-0.6B', 'all-MiniLM-L6-v2']

df['reduction'] = pd.Categorical(df['reduction'], categories=reduction_order, ordered=True)
df['cluster_method'] = pd.Categorical(df['cluster_method'], categories=cluster_order, ordered=True)
df['embedding'] = pd.Categorical(df['embedding'], categories=embedding_order, ordered=True)

pivot = df.pivot_table(
    index=['reduction', 'cluster_method'],
    columns='embedding',
    values='ARI'
)

pivot = pivot.sort_index()

pivot

/var/folders/1s/gzgswgnn7td1g02_q1hrvy3w0000gn/T/ipykernel_14257/2150432534.py:13: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(


embedding                         Qwen3-Embedding-0.6B  all-MiniLM-L6-v2
reduction cluster_method                                                
PHATE     Agglomerative                       0.476800          0.503058
          HDBSCAN                            -0.000139          0.028377
          Diffusion condensation              0.296899          0.080199
PCA       Agglomerative                       0.478989          0.263159
          HDBSCAN                             0.000018          0.000164
          Diffusion condensation              0.082603          0.077261
UMAP      Agglomerative                       0.490207          0.498970
          HDBSCAN                             0.428488         -0.000349
          Diffusion condensation              0.112346          0.091755